# Analyse des dépenses pharmaceutiques en France (2016–2025)
## Rapport de conclusions

---

**Source des données** : Open Medic / AMELI — Assurance Maladie  
**Période analysée** : 2016–2025  
**Méthode** : Statistiques descriptives + Régression OLS  

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
DATA_DIR    = Path('C:/Users/Enes/data')

def parse_euro(s):
    return (s.astype(str).str.strip()
             .str.replace('.','',regex=False)
             .str.replace(',','.',regex=False)
             .replace('','0').astype(float))

def load_year(annee):
    for pat in [f'OPEN_MEDIC_{annee}.zip', f'OPEN_MEDIC_{annee}.csv']:
        p = DATA_DIR / pat
        if p.exists():
            break
    else:
        return None
    df = pd.read_csv(p, sep=None, engine='python', encoding='latin-1')
    df.columns = df.columns.str.lower().str.strip()
    for col in ['l_atc1','l_cip13']:
        if col not in df.columns: df[col] = ''
    for col in ['rem','bse']:
        df[col] = parse_euro(df[col])
    df['annee'] = annee
    return df

# Chargement de toutes les années
frames = [load_year(a) for a in range(2016, 2026)]
df = pd.concat([f for f in frames if f is not None], ignore_index=True)

AGE_LABELS  = {0: '<20 ans', 20: '20–59 ans', 60: '60+ ans', 99: 'Inconnu'}
REG_LABELS  = {
    11: 'Île-de-France', 84: 'Auvergne-Rhône-Alpes', 93: 'PACA',
    75: 'Nouvelle-Aquitaine', 76: 'Occitanie', 32: 'Hauts-de-France',
    44: 'Grand Est', 52: 'Pays de la Loire', 53: 'Bretagne',
    28: 'Normandie', 24: 'Centre-Val de Loire', 27: 'Bourgogne-Franche-Comté',
    94: 'Corse', 1: 'Guadeloupe', 2: 'Martinique', 3: 'Guyane',
    4: 'La Réunion', 5: 'Mayotte', 99: 'Inconnu / Étranger'
}
df['age_label'] = df['age'].map(AGE_LABELS)
df['region']    = df['ben_reg'].map(REG_LABELS).fillna(df['ben_reg'].astype(str))

print(f'✓ Données chargées : {df.shape[0]:,} lignes | {df["annee"].nunique()} années')

---
## Partie I — Grandes tendances des dépenses pharmaceutiques (2016–2025)

### 1.1 Vue d'ensemble

La période 2016–2025 est marquée par une **croissance continue** des dépenses de remboursement  
de médicaments en France. Cette dynamique s'explique par trois facteurs structurels :

1. **Le vieillissement de la population** : la part des 60+ ans dans la consommation est en hausse constante
2. **L'innovation thérapeutique** : mise sur le marché de molécules coûteuses (oncologie, maladies rares)
3. **L'augmentation des maladies chroniques** : diabète, cardiovasculaire, troubles psychiatriques

In [ ]:
evo = df.groupby('annee').agg(
    boites        = ('boites', 'sum'),
    remboursement = ('rem',    'sum')
).reset_index()

# Taux de croissance cumulé
rem_2016 = evo.loc[evo['annee']==2016, 'remboursement'].values[0]
rem_last = evo['remboursement'].iloc[-1]
annee_last = evo['annee'].iloc[-1]
croissance = (rem_last / rem_2016 - 1) * 100

fig, axes = plt.subplots(1, 2)

axes[0].fill_between(evo['annee'], evo['boites']/1e6, alpha=0.3, color='steelblue')
axes[0].plot(evo['annee'], evo['boites']/1e6, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Évolution du volume (millions de boîtes)')
axes[0].set_xlabel('Année')
axes[0].set_xticks(range(2016, annee_last+1))
axes[0].tick_params(axis='x', rotation=45)

axes[1].fill_between(evo['annee'], evo['remboursement']/1e9, alpha=0.3, color='seagreen')
axes[1].plot(evo['annee'], evo['remboursement']/1e9, marker='o', color='seagreen', linewidth=2)
axes[1].set_title('Évolution des remboursements (milliards €)')
axes[1].set_xlabel('Année')
axes[1].set_xticks(range(2016, annee_last+1))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f'Croissance totale des remboursements 2016→{annee_last} : +{croissance:.1f}%')
print(f'Remboursements 2016  : {rem_2016/1e9:.2f} Md€')
print(f'Remboursements {annee_last} : {rem_last/1e9:.2f} Md€')

### 1.2 Le poids croissant des seniors

Les patients de **60 ans et plus** représentent la majorité des dépenses remboursées.  
Cette concentration reflète la **polymédication** liée aux pathologies chroniques de la vieillesse  
(cardiovasculaire, diabète, ostéoporose, douleur chronique).

In [ ]:
age_evo = (
    df[df['age'] != 99]
    .groupby(['annee', 'age_label'])['rem'].sum()
    .unstack('age_label').fillna(0)
    .reindex(columns=['<20 ans', '20–59 ans', '60+ ans'])
)

age_evo_pct = age_evo.div(age_evo.sum(axis=1), axis=0) * 100

age_evo_pct.plot(kind='area', alpha=0.6, figsize=(13, 5),
                 color=['#4CAF50','#2196F3','#FF9800'])
plt.title('Part des remboursements par tranche d\'âge (%) — 2016–2025')
plt.xlabel('Année')
plt.ylabel('%')
plt.legend(title='Tranche d\'âge')
plt.xticks(range(2016, annee_last+1), rotation=45)
plt.tight_layout()
plt.show()

print('Part des 60+ dans les remboursements :')
for yr in [2016, 2020, annee_last]:
    if yr in age_evo_pct.index:
        print(f'  {yr} : {age_evo_pct.loc[yr, "60+ ans"]:.1f}%')

---
## Partie II — Classes thérapeutiques les plus coûteuses

### 2.1 Classement par remboursement total

Les dépenses ne sont pas uniformément réparties entre classes thérapeutiques.  
Quelques classes concentrent l'essentiel des remboursements :

- **Système cardiovasculaire** : traitements hypertension, anticoagulants, statines — volume très élevé, prix unitaire modéré
- **Antinéoplasiques (oncologie)** : volume faible mais prix unitaire très élevé — poids croissant depuis 2018
- **Système nerveux** : antidépresseurs, antipsychotiques, antiépileptiques — progression liée aux maladies mentales
- **Système digestif et métabolisme** : antidiabétiques, IPP — forte hausse liée à l'épidémie de diabète de type 2

In [ ]:
atc_total = (
    df[df['l_atc1'] != '']
    .groupby('l_atc1')
    .agg(remboursement=('rem','sum'), boites=('boites','sum'))
    .sort_values('remboursement', ascending=False)
)
atc_total['cout_moyen_boite'] = atc_total['remboursement'] / atc_total['boites']
atc_total['part_pct'] = atc_total['remboursement'] / atc_total['remboursement'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

atc_total['remboursement'].sort_values().plot(
    kind='barh', ax=axes[0], color='seagreen')
axes[0].set_title('Remboursement cumulé 2016–2025 par classe ATC1')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e9:.1f}Md€'))

atc_total['cout_moyen_boite'].sort_values().plot(
    kind='barh', ax=axes[1], color='tomato')
axes[1].set_title('Coût moyen par boîte (€) — 2016–2025')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}€'))

plt.tight_layout()
plt.show()

print('Top 5 classes par remboursement total :')
print(atc_total[['remboursement','part_pct','cout_moyen_boite']]
      .head(5).rename(columns={
          'remboursement':'Total remboursé (€)',
          'part_pct':'Part (%)',
          'cout_moyen_boite':'Coût/boîte (€)'
      }).to_string())

### 2.2 L'oncologie : peu de volume, coût très élevé

Les **antinéoplasiques** se distinguent par un paradoxe :  
un volume de boîtes dispensées relativement faible, mais un **coût moyen par boîte**  
très supérieur aux autres classes.  

Ce phénomène est structurel : les thérapies ciblées et immunothérapies,  
approuvées massivement depuis 2015, ont des prix de lancement très élevés  
(souvent > 5 000 € / boîte), reflétant les coûts de R&D et les prix négociés  
avec le Comité Économique des Produits de Santé (CEPS).

In [ ]:
# Évolution du coût moyen/boîte par classe — les 5 plus chères
atc_annee = (
    df[df['l_atc1'] != '']
    .groupby(['annee','l_atc1'])
    .agg(rem=('rem','sum'), boites=('boites','sum'))
    .reset_index()
)
atc_annee['cout_boite'] = atc_annee['rem'] / atc_annee['boites']

top5_cheres = (
    atc_annee.groupby('l_atc1')['cout_boite'].mean()
    .nlargest(5).index
)

pivot = (atc_annee[atc_annee['l_atc1'].isin(top5_cheres)]
         .pivot(index='annee', columns='l_atc1', values='cout_boite'))

pivot.plot(marker='o', linewidth=2, figsize=(13, 6))
plt.title('Évolution du coût moyen par boîte — Top 5 classes les plus chères (€)')
plt.xlabel('Année')
plt.ylabel('€ / boîte')
plt.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.xticks(range(2016, annee_last+1), rotation=45)
plt.tight_layout()
plt.show()

---
## Partie III — Résultats de la régression OLS

### 3.1 Cadre d'analyse

La régression OLS vise à **quantifier l'effet marginal** de chaque facteur  
sur le montant remboursé par classe thérapeutique et par année.

**Modèle estimé :**

$$\text{Remboursement}_{i,t} = \alpha + \beta_1 \cdot \text{Année}_t + \beta_2 \cdot \text{Boîtes}_{i,t} + \sum_k \gamma_k \cdot \mathbb{1}[\text{ATC1} = k] + \varepsilon_{i,t}$$

Où $i$ désigne une classe ATC1 et $t$ une année.

### 3.2 Interprétation des coefficients

In [ ]:
# Charge les résultats OLS exportés par le notebook 04
ols_path = OUTPUTS_DIR / 'resultats_ols.csv'

if ols_path.exists():
    res = pd.read_csv(ols_path)
    metriques = pd.read_csv(OUTPUTS_DIR / 'metriques_ols.csv')

    print('=== MÉTRIQUES DU MODÈLE ===')
    print(metriques.to_string(index=False))
    print()

    print('=== VARIABLES SIGNIFICATIVES (p < 0.05) ===')
    sig = res[res['p-value'] < 0.05].sort_values('p-value')
    print(sig.to_string(index=False))
else:
    print('⚠ Fichier resultats_ols.csv introuvable.')
    print('  → Exécutez d\'abord le notebook 04_regression_ols.ipynb')

### 3.3 Ce que disent les résultats

#### Coefficient de l'année (tendance temporelle)
Le coefficient de la variable `annee` mesure l'augmentation annuelle moyenne des remboursements,  
**toutes choses égales par ailleurs** (même volume de boîtes, même classe thérapeutique).  
Un coefficient positif et significatif confirme une **dérive structurelle des prix** :  
les dépenses augmentent même à volume constant, signe d'une inflation du prix moyen par boîte.

#### Coefficient des boîtes (effet volume)
Le coefficient de `boites_total` capture la **sensibilité des remboursements au volume**.  
Il représente le remboursement marginal associé à une boîte supplémentaire délivrée.  
Un coefficient élevé indique des classes où le prix unitaire est fort.

#### Effets fixes ATC (dummies)
Les coefficients des dummies ATC mesurent l'**écart de dépenses** par rapport  
à la classe de référence (celle exclue par `drop_first=True`).  
Une dummy positive et significative signifie que cette classe génère structurellement  
plus de remboursements que la référence, à volume et année constants.

#### R² et qualité d'ajustement
Le R² indique la part de variance des remboursements expliquée par le modèle.  
Un R² élevé (> 0.80) valide la pertinence des variables retenues.  
Le gain de R² entre la régression simple (année seule) et le modèle complet  
mesure la contribution des dummies ATC et du volume.

In [ ]:
if ols_path.exists():
    # Visualisation des coefficients ATC significatifs
    atc_sig = res[
        res['Variable'].str.startswith('ATC_') &
        (res['p-value'] < 0.05)
    ].sort_values('Coefficient')

    if not atc_sig.empty:
        colors = ['tomato' if c < 0 else 'steelblue' for c in atc_sig['Coefficient']]
        atc_sig.set_index('Variable')['Coefficient'].plot(
            kind='barh', color=colors, figsize=(12, 6))
        plt.axvline(0, color='black', linewidth=0.8)
        plt.title('Coefficients OLS significatifs — Effets fixes ATC')
        plt.xlabel('Coefficient (€)')
        plt.tight_layout()
        plt.show()
    else:
        print('Aucune dummy ATC significative à afficher.')

---
## Partie IV — Limites du modèle et pistes d'amélioration

### 4.1 Limites identifiées

#### Limites méthodologiques

| Limite | Description | Impact |
|--------|-------------|--------|
| **Agrégation** | Les données sont agrégées par classe ATC × année, ce qui efface l'hétérogénéité intra-classe | Biais d'agrégation — les effets individuels sont masqués |
| **Hétéroscédasticité** | Les classes oncologiques ont une variance des résidus bien plus élevée que les autres | Intervalles de confiance biaisés (→ utiliser erreurs robustes HC3) |
| **Multicolinéarité potentielle** | Forte corrélation entre année et certaines classes ATC dont les remboursements suivent une tendance linéaire | Gonflement des std errors de ces variables |
| **Linéarité assumée** | L'OLS suppose une relation linéaire entre X et Y | Si la relation est log-linéaire ou non-linéaire, le modèle est mal spécifié |
| **Absence de variables** | Pas de données sur les prix unitaires, les politiques de remboursement (délistages, baisses de prix CEPS), la démographie | Variables omises → biais sur les coefficients estimés |

#### Limites des données Open Medic

- **Données de remboursement, pas de consommation** : un médicament non remboursé (ou partiellement) est sous-représenté
- **Pas de données hospitalières** : l'hôpital représente ~30% des dépenses pharmaceutiques totales mais est absent d'Open Medic
- **Agrégation par tranches** : les variables `age` et `sexe` sont en catégories larges, pas en valeur continue
- **Ruptures méthodologiques** : le format a changé entre 2015 et 2016 (passage de 15 à 21 colonnes)

---

### 4.2 Pistes d'amélioration

#### Amélioration du modèle économétrique

**1. Transformation logarithmique**  
Les dépenses pharmaceutiques suivent souvent une distribution log-normale.  
Estimer `log(Remboursement)` réduit l'hétéroscédasticité et améliore l'ajustement :
```python
Y_log = np.log1p(df_model['rem_total'])
model_log = sm.OLS(Y_log, X_multi).fit()
```

**2. Données de panel (effets fixes)**  
Avec 10 années × N classes ATC, une régression sur données de panel serait plus rigoureuse :
```python
from linearmodels import PanelOLS
# Effets fixes classe ATC → élimine l'endogénéité intra-classe
```

**3. Variables de contrôle manquantes**  
Enrichir avec :
- Prix unitaire moyen par CIP13 (source : base Médicament ANSM)
- Indicateur de générication (TOP_GEN)
- Données démographiques régionales (INSEE)
- Indicateurs épidémiologiques (prévalence des pathologies)

**4. Modèles non-linéaires**  
Pour capturer des effets de seuil ou des interactions :
- **Random Forest Regressor** : importance des variables sans hypothèse de linéarité
- **Gradient Boosting (XGBoost)** : prédiction plus précise
- **Régression de Poisson** : si la variable dépendante est un comptage

---

### 4.3 Synthèse des recommandations

> **À court terme** : réestimer avec erreurs robustes HC3 (déjà fait en notebook 04) et tester la spécification log-log.

> **À moyen terme** : coupler Open Medic avec la base SNDS (Système National des Données de Santé) pour disposer de données individuelles anonymisées, permettant une modélisation plus fine.

> **Pour la prédiction** : construire un modèle de prévision des dépenses 2026–2030 par classe ATC en intégrant les projections démographiques INSEE et le pipeline de mise sur le marché ANSM.

---

## Conclusion générale

Cette analyse confirme trois résultats robustes :

1. **Les dépenses pharmaceutiques remboursées croissent structurellement** sur la période 2016–2025, portées par le vieillissement et l'innovation coûteuse en oncologie.

2. **La concentration est forte** : quelques classes thérapeutiques (cardiovasculaire, oncologie, système nerveux) représentent la majorité des remboursements.

3. **Le modèle OLS valide** l'effet significatif de la tendance temporelle et du volume sur les remboursements, mais ses hypothèses (linéarité, homoscédasticité) sont partiellement violées — justifiant le recours à des erreurs robustes et à des spécifications alternatives.

---
*Analyse réalisée avec Python (pandas, statsmodels, matplotlib) — données Open Medic / AMELI*